# ACIT4620 Group 1 — Notebook 11: Computational Efficiency (RQ4)

**Submitted proposal RQ4:** What additional *training, inference, and memory costs* are associated with the neuro-fuzzy classifier, and how does the number of fuzzy rules influence the trade-off between predictive performance and computational efficiency?

**Important:** This notebook compares **saved weights** from E2a-50, E3 and E4. It does not retrain the models for predictive purposes. The original 1,458-image test set stays locked. We distinguish: **historical training durations**, a **matched training-step microbenchmark**, **head-only inference**, **CNN + head inference**, and optionally **real JPEG-to-classification latency**. These are not interchangeable metrics.

## 1. Environment and benchmark setup

Use the working E3/Notebook 10 Python `.venv` as the VS Code kernel. CPU works; using CUDA changes all timing results. By default we benchmark **real validation images** from Notebook 10; if no original JPEGs exist, explicitly set `IMAGE_MODE='synthetic'` and do not describe that as real-image performance.

In [1]:
import sys
print(sys.executable)

c:\Users\grela\OneDrive\Desktop\OSLOMET\COMPUTATIONAL INTELLIGENCE\FOR FINAL REPORT\CODE\ACIT4620_E3_VSCode_Complete\.venv\Scripts\python.exe


In [2]:
import sys, os, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch

PROJECT_ROOT=Path.cwd().resolve()
if not (PROJECT_ROOT/'efficiency_benchmark.py').is_file():
    nested=PROJECT_ROOT/'ACIT4620_11_Efficiency_VSCode'
    if (nested/'efficiency_benchmark.py').is_file():PROJECT_ROOT=nested
assert (PROJECT_ROOT/'efficiency_benchmark.py').is_file(), 'Open the INNER Notebook 11 project folder.'
if str(PROJECT_ROOT) not in sys.path:sys.path.insert(0,str(PROJECT_ROOT))

from efficiency_benchmark import (load_models, model_stats, historical_training,
    benchmark_head_only, prepare_real_images, benchmark_tensor_to_logit,
    benchmark_image_file_pipeline, training_step_microbenchmark, finalize, SELECTED)

# Set to the root of your existing GitHub checkout if automatic detection fails:
# Example: r'C:\AI\G1-NB10'
TEAM_REPO = None

# Set to the folder containing HAM10000_images_part_1 and HAM10000_images_part_2,
# e.g. the data/raw directory from the successful Notebook 10:
RAW_IMAGES_ROOT = None

IMAGE_MODE = 'real'    # 'real' (recommended), or explicit 'synthetic' shape-matched timing
DEVICE = 'cpu'        # keep CPU for comparability with E3/E4 local tests; CUDA optional
CPU_THREADS = 4
REAL_VALIDATION_IMAGES = 16

print('Python:',sys.executable,'| Torch:',torch.__version__)
print('Notebook 11 root:',PROJECT_ROOT,'| Mode:',IMAGE_MODE,'| Device:',DEVICE)

Python: c:\Users\grela\OneDrive\Desktop\OSLOMET\COMPUTATIONAL INTELLIGENCE\FOR FINAL REPORT\CODE\ACIT4620_E3_VSCode_Complete\.venv\Scripts\python.exe | Torch: 2.15.0+cpu
Notebook 11 root: C:\Users\grela\OneDrive\Desktop\OSLOMET\COMPUTATIONAL INTELLIGENCE\FOR FINAL REPORT\CODE\Notebook_11_Computational_Efficiency_VSCode_Complete\ACIT4620_11_Efficiency_VSCode | Mode: real | Device: cpu


## 2. Integrity gate — original CNN, E3 and E4 checkpoints

Loads E2a-50 backbone and **six E3 / nine E4 checkpoints**. Checks the full split fingerprint, class order, CNN checkpoint hash, and equal training-only feature scalers. If E3 weights are not found, point `TEAM_REPO` at your cloned merged GitHub repository. **Do not disable these checks.**

In [3]:
session=load_models(PROJECT_ROOT,team_repo=TEAM_REPO,device=DEVICE,cpu_threads=CPU_THREADS)
print('Models:',len(session.heads)+1,'(1 CNN classifier + 6 E3 + 9 ANFIS)')
print('Saved original checkpoint hashes:',len(session.checksum))
print('Test partition: LOCKED')

PASS: original CNN, 6 E3 and 9 E4 checkpoints loaded; split, CNN SHA, scaler verified.
GitHub checkout: C:\AI\G1-NB10 | device: cpu
Locked test images/labels were NOT accessed.
Models: 16 (1 CNN classifier + 6 E3 + 9 ANFIS)
Saved original checkpoint hashes: 17
Test partition: LOCKED


## 3. Parameter, checkpoint and *historical* training-time audit

These training times are **the historical records from E3/E4**. They were obtained with different early-stopping schedules and must **not** be interpreted as a controlled head-cost comparison. The CNN baseline was trained separately in Kaggle and is not assigned a made-up total duration.

In [4]:
parameters=model_stats(session)
historical=historical_training(session)
display(parameters[parameters.model.isin(SELECTED)].round(4))
display(historical.groupby('family').agg(
    seeds=('seed','count'),recorded_training_s_mean=('train_seconds','mean'),
    recorded_training_s_sd=('train_seconds','std'),epochs_mean=('epochs_completed','mean')
).round(4))

,model,backbone_parameters,head_parameters,additional_parameters_vs_original_cnn,effective_deployed_parameters,head_parameter_memory_mb_f32,checkpoint_size_mb
0,cnn_e2a50,4007548,8967,0,4016515,0.0342,15.6113
1,linear_seed42,4007548,8967,0,4016515,0.0342,0.0365
4,mlp_seed42,4007548,10767,1800,4018315,0.0411,0.0444
8,anfis_r4_seed52,4007548,10564,1597,4018112,0.0403,0.0438
10,anfis_r8_seed42,4007548,10880,1913,4018428,0.0415,0.0450
14,anfis_r16_seed52,4007548,11512,2545,4019060,0.0439,0.0474


,seeds,recorded_training_s_mean,recorded_training_s_sd,epochs_mean
family,,,,
anfis_r16,3,4.9657,1.0532,36.0000
anfis_r4,3,3.9120,0.5220,40.3333
anfis_r8,3,4.9283,0.8281,44.0000
linear,3,4.2700,0.5631,70.3333
mlp,3,3.6433,0.9034,48.3333


## 4. Matched head-only inference cost — all 16 models

The inputs below are **fixed synthetic shape-matched feature tensors**, not classified lesions. This isolates head compute, permits comparison across all 15 neural/fuzzy head checkpoints plus the CNN's linear classifier, and does *not* measure image preprocessing or CNN feature extraction. Timing includes warmup and repeated passes.

head_cost=benchmark_head_only(
    session, batch_sizes=(1,32,128), repeats=16,warmups=4,all_seeds=True
)
print('Head-only measurement rows:',len(head_cost))
display(head_cost[(head_cost.batch_size==1)&head_cost.model.isin(SELECTED)][
    ['model','mean_ms_per_image','median_batch_ms','p95_batch_ms',
     'throughput_images_per_second','device']
].round(5))

In [12]:

head_cost = benchmark_head_only(
    session,
    batch_sizes=(1, 32, 128),
    repeats=100,
    warmups=15,
    all_seeds=True
)

print("Head-only measurement rows:", len(head_cost))

display(
    head_cost[
        (head_cost.batch_size == 1) &
        head_cost.model.isin(SELECTED)
    ][[
        "model",
        "mean_ms_per_image",
        "median_batch_ms",
        "p95_batch_ms",
        "throughput_images_per_second",
        "device"
    ]].round(5)
)


Head-only measurement rows: 48


,model,mean_ms_per_image,median_batch_ms,p95_batch_ms,throughput_images_per_second,device
0,cnn_e2a50,0.01297,0.01130,0.02420,77089.11413,cpu
1,linear_seed42,0.00858,0.00840,0.00901,116509.37976,cpu
4,mlp_seed42,0.03806,0.03775,0.05119,26275.68441,cpu
8,anfis_r4_seed52,0.06319,0.06150,0.07102,15824.28712,cpu
10,anfis_r8_seed42,0.06410,0.06200,0.07222,15600.86741,cpu
14,anfis_r16_seed52,0.06370,0.06250,0.07276,15697.60144,cpu


## 5. Prepare identical model inputs for all end-to-end comparisons

In `real` mode: select **only** the previously declared Notebook 10 validation image IDs and preprocess using the same ImageNet transform. The held-out test is not opened. `synthetic` mode is an explicit fallback for hardware timing, with that limitation included in exported metadata. Image decoding and preprocessing are measured separately in Section 7.

In [13]:
if IMAGE_MODE == 'real':
    benchmark_images,validation_jpegs=prepare_real_images(
        session,raw_override=RAW_IMAGES_ROOT,max_images=REAL_VALIDATION_IMAGES
    )
    input_label='real_validation_preprocessed_tensors'
    print('Real validation images:',len(validation_jpegs))
    print('First validation image ID:',validation_jpegs[0][0])
elif IMAGE_MODE == 'synthetic':
    g=torch.Generator().manual_seed(20261109)
    benchmark_images=torch.randn(
        REAL_VALIDATION_IMAGES,3,224,224,generator=g
    ).to(session.device)
    validation_jpegs=None
    input_label='synthetic_shape_matched_image_tensors'
    print('WARNING: TIMINGS USE SYNTHETIC IMAGE TENSORS, NOT ACTUAL HAM10000 PHOTOGRAPHS.')
else:
    raise ValueError('IMAGE_MODE must be real or synthetic')
assert benchmark_images.shape==(REAL_VALIDATION_IMAGES,3,224,224)
print('Common tensor batch shape:',tuple(benchmark_images.shape))

Real validation images: 16
First validation image ID: ISIC_0032152
Common tensor batch shape: (16, 3, 224, 224)


## 6. Matched CNN + head inference — tensor-to-class logits

All six selected models process identical input tensors through the same EfficientNet-B0 backbone. Includes the train-only scaler and full ANFIS inference, **but excludes image file I/O and decoding**. Compare the same batch size, device, thread count and run repetitions. These outputs are computational timings, **not validation accuracy results**.

full_cost=benchmark_tensor_to_logit(
    session,benchmark_images,mode=input_label,
    batch_sizes=(1,8),repeats=8,warmups=2
)
display(full_cost[[
    'model','batch_size','input_type','median_batch_ms',
    'mean_ms_per_image','p95_batch_ms','throughput_images_per_second'
]].round(3))

In [14]:

full_cost = benchmark_tensor_to_logit(
    session,
    benchmark_images,
    mode=input_label,
    batch_sizes=(1, 8),
    repeats=30,
    warmups=10
)

display(
    full_cost[[
        "model",
        "batch_size",
        "input_type",
        "median_batch_ms",
        "mean_ms_per_image",
        "p95_batch_ms",
        "throughput_images_per_second"
    ]].round(3)
)


,model,batch_size,input_type,median_batch_ms,mean_ms_per_image,p95_batch_ms,throughput_images_per_second
0,cnn_e2a50,1,real_validation_preprocessed_tensors,9.669,10.380,12.784,96.339
1,linear_seed42,1,real_validation_preprocessed_tensors,9.459,9.740,11.429,102.666
2,mlp_seed42,1,real_validation_preprocessed_tensors,9.451,9.708,11.615,103.009
3,anfis_r4_seed52,1,real_validation_preprocessed_tensors,9.510,9.709,10.678,103.001
4,anfis_r8_seed42,1,real_validation_preprocessed_tensors,9.595,9.885,11.429,101.165
5,anfis_r16_seed52,1,real_validation_preprocessed_tensors,9.551,9.665,10.772,103.463
6,cnn_e2a50,8,real_validation_preprocessed_tensors,49.963,6.393,58.049,156.433
7,linear_seed42,8,real_validation_preprocessed_tensors,50.553,6.393,62.944,156.414
8,mlp_seed42,8,real_validation_preprocessed_tensors,49.027,6.524,64.951,153.283
9,anfis_r4_seed52,8,real_validation_preprocessed_tensors,48.487,6.249,59.496,160.032


## 7. Optional real JPEG → classification latency

This **includes** reading the *same validation JPEGs*, ImageNet preprocessing, CNN inference and the classifier head. It is deliberately separate from the model-only latency. Skipped only when synthetic inputs are selected.

if IMAGE_MODE == 'real':
    raw_cost=benchmark_image_file_pipeline(
        session, validation_jpegs,batch_sizes=(1,8),repeats=6,warmups=2
    )
    display(raw_cost[['model','batch_size','mean_ms_per_image',
                      'p95_batch_ms','throughput_images_per_second']].round(3))
else:
    raw_cost=pd.DataFrame()
    print('SKIPPED: real JPEG-to-prediction latency requires original validation images.')

In [15]:

if IMAGE_MODE == "real":

    raw_cost = benchmark_image_file_pipeline(
        session,
        validation_jpegs,
        batch_sizes=(1, 8),
        repeats=30,
        warmups=10
    )

    display(
        raw_cost[[
            "model",
            "batch_size",
            "mean_ms_per_image",
            "p95_batch_ms",
            "throughput_images_per_second"
        ]].round(3)
    )

else:
    raw_cost = pd.DataFrame()
    print(
        "SKIPPED: real JPEG-to-prediction latency "
        "requires original validation images."
    )


,model,batch_size,mean_ms_per_image,p95_batch_ms,throughput_images_per_second
0,cnn_e2a50,1,14.208,16.682,70.382
1,linear_seed42,1,13.636,15.123,73.333
2,mlp_seed42,1,14.123,16.736,70.809
3,anfis_r4_seed52,1,13.754,15.649,72.709
4,anfis_r8_seed42,1,14.200,16.070,70.422
5,anfis_r16_seed52,1,13.562,14.108,73.734
6,cnn_e2a50,8,10.961,97.941,91.232
7,linear_seed42,8,11.022,101.637,90.727
8,mlp_seed42,8,11.038,101.708,90.600
9,anfis_r4_seed52,8,11.344,111.706,88.156


## 8. Matched gradient-step microbenchmark — training cost proxy

Clones existing trained head weights and measures a standard forward + backward + optimizer step on synthetic features and simulated labels. **These steps do not train or select a new classifier for the project.** Only use to describe approximate relative per-step costs, with the historical training durations separately in Section 3.

micro=training_step_microbenchmark(
    session,batch_size=128,steps=8,warmups=2
)
display(micro[['model','mean_training_step_ms','p95_training_step_ms',
               'train_step_samples_per_second']].round(3))

In [16]:

micro = training_step_microbenchmark(
    session,
    batch_size=128,
    steps=50,
    warmups=10
)

display(
    micro[[
        "model",
        "mean_training_step_ms",
        "p95_training_step_ms",
        "train_step_samples_per_second"
    ]].round(3)
)


,model,mean_training_step_ms,p95_training_step_ms,train_step_samples_per_second
0,linear_seed42,0.396,0.563,323591.870
1,mlp_seed42,0.640,0.823,199869.460
2,anfis_r4_seed52,0.806,0.901,158832.186
3,anfis_r8_seed42,0.793,0.907,161371.252
4,anfis_r16_seed52,0.912,1.023,140365.960


## 9. Figures, provenance, backup and scientific limitations

Generate tables, plots and a complete ZIP. Interpretation must state CPU threads, hardware, batch size, whether original or synthetic images were used, and whether measured memory is GPU peak or CPU RSS snapshots. Re-running on a different device requires an independent result set.

zip_file=finalize(
    session, IMAGE_MODE,head_cost,full_cost,historical,parameters,micro,raw=raw_cost
)
print('Generated figures:',sorted(p.name for p in (session.out/'figures').glob('*.png')))
print('Saved CSV files:',sorted(p.name for p in session.out.glob('*.csv')))
print('Complete ZIP:',zip_file)

In [17]:

zip_file = finalize(
    session,
    IMAGE_MODE,
    head_cost,
    full_cost,
    historical,
    parameters,
    micro,
    raw=raw_cost
)

print(
    "Generated figures:",
    sorted(p.name for p in (session.out / "figures").glob("*.png"))
)

print(
    "Saved CSV files:",
    sorted(p.name for p in session.out.glob("*.csv"))
)

print("Complete ZIP:", zip_file)


PASS: RQ4 outputs saved: C:\Users\grela\OneDrive\Desktop\OSLOMET\COMPUTATIONAL INTELLIGENCE\FOR FINAL REPORT\CODE\Notebook_11_Computational_Efficiency_VSCode_Complete\ACIT4620_11_Efficiency_VSCode\outputs\ACIT4620_11_Efficiency_Results
Backup ZIP: C:\Users\grela\OneDrive\Desktop\OSLOMET\COMPUTATIONAL INTELLIGENCE\FOR FINAL REPORT\CODE\Notebook_11_Computational_Efficiency_VSCode_Complete\ACIT4620_11_Efficiency_VSCode\outputs\ACIT4620_11_Efficiency_Results.zip size MB: 0.27
TEST SPLIT REMAINS LOCKED. These are local cost measurements, not test results.
Generated figures: ['01_head_parameters.png', '02_head_only_latency.png', '03_full_model_latency.png', '04_train_step_cost.png', '05_historical_training_times_caution.png']
Saved CSV files: ['end_to_end_jpg_to_logit.csv', 'end_to_end_tensor_to_logit.csv', 'head_only_inference.csv', 'historical_training_times_NOT_matched.csv', 'parameter_and_checkpoint_cost.csv', 'training_step_microbenchmark.csv']
Complete ZIP: C:\Users\grela\OneDrive\Desk

## 10. How to interpret RQ4 and proceed

1. **Incremental parameters:** compare E3/ANFIS head sizes and effective CNN-plus-head parameter counts, not only total CNN parameters.
2. **Cost versus rule count:** compare ANFIS-4, ANFIS-8, ANFIS-16 under matched batch sizes and device; increase in rules does not guarantee improved macro-F1 (see Notebook 09).
3. **Inference:** distinguish head-only from CNN-plus-head and JPEG-inclusive. Because all heads share the CNN, full image cost may be dominated by the backbone.
4. **Training:** historical end-to-end training durations and the new standardized gradient-step microbenchmark are distinct studies; do not conflate them.
5. **Memory:** report GPU incremental peak allocated bytes only if CUDA; a CPU process RSS change is not a model-specific peak measurement.
6. **Report:** include `parameter_and_checkpoint_cost.csv`, `head_only_inference.csv`, `end_to_end_tensor_to_logit.csv`, `training_step_microbenchmark.csv`, the provenance file and relevant figures. Use Notebook 09's accuracy/seed results for the performance side of the trade-off.

After measuring, preserve the output ZIP and place the complete experiment under `experiments/e7_computational_efficiency/` in a separate GitHub PR. **No final locked-test evaluation is performed here.**